<a href="https://colab.research.google.com/github/cankat-byte/DATA_SCIENCE_project/blob/main/MARPOL_RAG_CHATBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1.Hücre - Hazırlık

In [ ]:
# Gerekli tüm güncel ve uyumlu paketler
!pip install -q -U langchain langchain-community langchain-groq langchain-huggingface langchain-chroma chromadb pypdf streamlit


2.Hücre - Kernel yeniden başlatma ( Çalışma zamanı yeniden başlatmak için. DİKKAT sadece hazırlık aşaması sonrası çalıştır)

In [ ]:
# ⚠️ DİKKAT: Bu hücreyi SADECE paket kurulumu bittiğinde 1 KERE çalıştır.

import IPython
IPython.display.display(IPython.display.HTML("<script>google.colab.kernel.restart()</script>"))


3.Hücre - Groq API anahtarı tanımlama

In [ ]:
import os
from google.colab import userdata

# Colab sol panelindeki 🔑 simgesine tıklayıp "GROQ_API_KEY" adıyla anahtarı ekle
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')


**4.Hücre - Kütüphanelerin Yüklenmesi** ( bu adımı gereksiz olacağı için atladım , streamlit kendi hücresinde zaten kütüphaneleri okuyacak / colab içinde arayüz oluşturacağımda başka bir hücrede tekrar gerekli kütüphaneleri yazacğım)

5.Hücre - GROQ Kurulum (uygun model seçimi)

In [ ]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.1)



6.Hücre - PDF parçalama / Veritabanı kayıt

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# ==========================================
# 1. PDF Belgesini Yükleme
# ==========================================
pdf_yolu = "/content/sozlesme.pdf"

if not os.path.exists(pdf_yolu):
    raise FileNotFoundError(f"HATA: '{pdf_yolu}' konumunda PDF dosyası bulunamadı! Lütfen Colab'e yükleyin.")

loader = PyPDFLoader(pdf_yolu)
documents = loader.load()

# ==========================================
# 2. Metni Parçalara Bölme
# ==========================================
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

# ==========================================
# 3. Vektör Veritabanı ve Retriever Oluşturma
# ==========================================
# Güncel kütüphane üzerinden gömme (embedding) modeli çağrılır
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Bellek içi (InMemory) vektör deposu splits oluştuktan sonra beslenir
vectorstore = InMemoryVectorStore.from_documents(splits, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})  # En alakalı 3 parçayı getirir

# ==========================================
# 4. Prompt (Şablon) Tanımlama
# ==========================================
prompt = ChatPromptTemplate.from_template("""Aşağıdaki bağlamı (context) kullanarak soruyu cevapla:

Bağlam:
{context}

Soru: {input}

Cevap:""")

# ==========================================
# NOT: Buradan sonra bir LLM (Örn: OpenAI, Groq)
# modelini zincire (rag_chain) bağlama
# ==========================================
print(f"Başarılı! {len(splits)} metin parçası hafızadaki vektör veritabanına işlendi.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Başarılı! 1396 metin parçası hafızadaki vektör veritabanına işlendi.


7.Hücre - Parçaların okunabilirliğinin kontrolü (ilk 3 parça kontrol)

In [ ]:
# Üretilen parçalardan ilk 3 tanesini ekrana yazdırarak kontrol edelim
for i, chunk in enumerate(splits[:3]):
    print(f"=== PARÇA {i+1} ===")
    print(f"📄 Kaynak Sayfa: {chunk.metadata.get('page', 0) + 1}") # Sayfa indeksi 0'dan başlar, +1 ekliyoruz
    print(f"📏 Karakter Uzunluğu: {len(chunk.page_content)}")
    print("📝 İÇERİK:")
    print(chunk.page_content)
    print("=" * 30 + "\n")


=== PARÇA 1 ===
📄 Kaynak Sayfa: 1
📏 Karakter Uzunluğu: 750
📝 İÇERİK:
T.C . 
Resmî Gazete 
Kuruluş Tarihi : ( 7 Teşrinievvel 1336 ) — 7 Ekim 1920 
Yönetim ve yazı işleri için 
Başbakanlık Mevzuatı Geliştirme ve 
Yayın Genel Müdürlüğüne 
başvurulur. 
24 Haziran 1990 
PAZAR Sayı : 20558 
YÜRÜTME VE İDARE BOLUMU 
Milletlerarası Sözleşme 
Karar Sayısı ; 90/441 
Ekli "Denizlerin Gemiler Tarafından Kirletilmesinin önlenmesine Ait Uluslararası Söz­
leşme (MARPOL-73 Sözleşmesi)", Sözleşmeyi tadil eden Protokol (MARPOL-78 Protokolü) 
ve anılan protokole ilişkin I, II ve V numaralı ekler ile 1984, 1985 ve 1987 tarihlerinde yapılan 
değişikliklerine katılmamız ve 13/9/1989 tarihli ve 89/14547 sayılı Kararnamenin yürürlükten 
kaldırılması; Dışişleri Bakanlığı'nın 30/3/1990, 12/4/1990 tarihli ve 1750, 2022 sayılı yazıları

=== PARÇA 2 ===
📄 Kaynak Sayfa: 1
📏 Karakter Uzunluğu: 786
📝 İÇERİK:
değişikliklerine katılmamız ve 13/9/1989 tarihli ve 89/14547 sayılı Kararnamenin yürürlükten 
kaldırılması; Dı

8.Hücre - Vektör veri tabanı ve Embedding Kontrolü

In [ ]:
# 1. InMemory veritabanındaki toplam parça (doküman) sayısını kontrol edelim
toplam_kayit = len(vectorstore.store.keys())
print(f"Veritabanındaki Toplam Vektör Sayısı: {toplam_kayit}\n")

# 2. Örnek bir sorunun nasıl sayısallaştırıldığını (Embedding) görelim
ornek_soru = "Sözleşmenin süresi ne kadar?"
soru_vektoru = embeddings.embed_query(ornek_soru)

print("🔢 Soru Metninin Sayısal (Embedding) Karşılığı:")
print(f"• Vektör Boyutu (Boyut Sayısı): {len(soru_vektoru)}")
print(f"• İlk 5 Sayısal Değer: {soru_vektoru[:5]} ... (Devam ediyor)")
print("-" * 50 + "\n")

# 3. Veritabanının benzerlik aramasını (En Basit Yöntemle) test edelim
print(f"🔍 '{ornek_soru}' Sorusu İçin En Yakın 2 Parçanın Bulunması:")
# En kararlı ve basit arama fonksiyonu
en_yakin_parcalar = vectorstore.similarity_search(ornek_soru, k=2)

for i, doc in enumerate(en_yakin_parcalar):
    print(f"=== EŞLEŞME {i+1} ===")
    print(f"📄 Kaynak Sayfa: {doc.metadata.get('page', 0) + 1}")
    print(f"📝 İlgili Metin İçeriği:\n{doc.page_content}")
    print("=" * 30 + "\n")


Veritabanındaki Toplam Vektör Sayısı: 1396

🔢 Soru Metninin Sayısal (Embedding) Karşılığı:
• Vektör Boyutu (Boyut Sayısı): 384
• İlk 5 Sayısal Değer: [-0.11612400412559509, 0.1152413859963417, -0.0821220800280571, 0.0191726665943861, -0.11063505709171295] ... (Devam ediyor)
--------------------------------------------------

🔍 'Sözleşmenin süresi ne kadar?' Sorusu İçin En Yakın 2 Parçanın Bulunması:
=== EŞLEŞME 1 ===
📄 Kaynak Sayfa: 64
📝 İlgili Metin İçeriği:
Eklenti 11 
BU BELGE 
1. Geminin Sözleşme Ek 1 Kural L hükmüne uygun olarak sörveye 
tabı tutulduğunu;ve 
2. Bu sörvey sonunda geminin yapısı, teçhizatı,sistemleri, bağlantıları 
malzemesi ve durumunun her bakımdan yeterli olduğunu ve 
Sözleşmenin Ek 1 hükümlerine uygun olduğunu onaylar. 
Bu Belge, j... 
tarihine kadar, Sözleşmenin Ek 1, Kural 4'üne göre yapılacak sörveylere 
bağlı kalmak koşulu ile geçerlidir. 
, de düzenlenmiştir. 
(belgenin verildiği yer) 
19.. 
(veriliş Tarihi) 
(Belgeyi veren resmi yetkilinin 
İmzası) 
(Yetki


**Eşleşme 1 Neden Seçildi? (Güçlü Benzerlik)**

Embedding modeli (yapay zeka) bu metni doğrudan zaman, geçerlilik ve süre kavramlarıyla eşleştirmiştir. Metnin içinde geçen şu ifadeler sorunuzun koordinatlarına çok yakındır:

• "... tarihine kadar ... geçerlidir" cümlesi doğrudan bir süreyi, son kullanma tarihini ve müddeti belirtir.

• "Sözleşmenin Ek 1... hükümlerine uygun olduğunu onaylar" ifadesi aradığınız "sözleşme" kelimesini içerir.

• Alt kısımdaki "Veriliş Tarihi" ve "19.." gibi alanlar zaman haritasında sorunuzla yan yana durur.

Yapay zeka, kelimeler birebir aynı olmasa bile bu metnin "bir şeyin süresinden ve geçerliliğinden" bahsettiğini anlayıp 1. sıraya koymuştur.

**Eşleşme 2 Neden Seçildi? (Zayıf Eşleşme / Hata)**

Vektör veri tabanları siz ne sorarsanız sorun, elindeki en yakın sonuçları getirmek zorundadır. 2. metin aslında bir gemi tablosudur ve süreyle ilgisi yoktur. Ancak sistem bunu şu yüzden 2. sıraya almıştır:

• Metnin başında "Gemide Tamamlanacak" ifadesi yer alıyor. Model, "tamamlanmak" eylemini zamansal bir süreç (bir sürenin bitmesi) olarak yorumlamış olabilir.

• Başka yakın parça bulamadığı için en azından yapısal olarak benzer olan bu dokümanı listeye eklemiştir.

9.Hücre - TEST (40 adet soru & cevap)

In [ ]:
import time
from langchain_groq import ChatGroq

# ==========================================
# 1. 40 Adet Soru Listesi
# ==========================================
altin_veri_seti = [
    {"soru": "Metnin başında hangi makamın bildirim yapması gerektiği belirtiliyor?", "dogru_cevap": "Genel Sekreter."},
    {"soru": "Mahkeme kaç üyeden oluşur?", "dogru_cevap": "Üç üyeden oluşur."},
    {"soru": "Taraflar kaç hakem atar?", "dogru_cevap": "Her taraf bir hakem atar."},
    {"soru": "Üçüncü hakemin atanması kimlerin mutabakatıyla gerçekleşir?", "dogru_cevap": "İki hakemin ortak mutabakatı ile."},
    {"soru": "İkinci hakemin tayininden sonra kaç gün içinde başkan tayin edilemezse ne olur?", "dogru_cevap": "60 gün içinde başkan tayin edilemezse taraflar bilgi ilave eder."},
    {"soru": "Tamamlayıcı raporun gönderilmesi hangi şartlara bağlıdır?", "dogru_cevap": "Protokol hükümlerine uygun olarak."},
    {"soru": "Tamamlayıcı raporun ilk kısmı ne zaman gönderilmelidir?", "dogru_cevap": "Gelişmeler olduğunda ilk rapor gönderilmelidir."},
    {"soru": "Olaydan etkilenen devletlerin ek bilgi istekleri nasıl karşılanır?", "dogru_cevap": "Eksiksiz olarak cevaplandırılır."},
    {"soru": "Protokolün hangi maddeleri hükümlerine uygun olarak Komisyonun kurulması isteniyor?", "dogru_cevap": "1. ve 3. maddeler."},
    {"soru": "Komisyonun kurulması için ne kadar süre veriliyor?", "dogru_cevap": "En erken bir fırsatta, 30.02.1974 tarihine kadar."},
    {"soru": "Komisyonun oy veren tarafların kaçı çoğunluğu ile listeyi kabul etmesi bekleniyor?", "dogru_cevap": "Üçte iki çoğunluk."},
    {"soru": "Komisyonun hangi kurumlarla istişare etmesi öneriliyor?", "dogru_cevap": "Uluslararası uzman kuruluşlarla."},
    {"soru": "Metinde 'Tanzim' kelimesi hangi bağlamda kullanılmıştır?", "dogru_cevap": "Listeyi düzenleme ve tanzim etme."},
    {"soru": "'Tanzim' kelimesinin Türkçe karşılığı nedir?", "dogru_cevap": "Düzenleme, hazırlama."},
    {"soru": "Metinde 'Tanzim' kelimesi kaç kez geçiyor?", "dogru_cevap": "2 kez."},
    {"soru": "'Tanzim' kelimesi hangi cümlede 'kopyalarının' ile birlikte geçiyor?", "dogru_cevap": "“Liste tanzim edilir edilmez kopyalarının Teşkilâtın Genel Sekreteri”"},
    {"soru": "Gemideki 'slop' tankı nedir ve hangi amaçla kullanılır?", "dogru_cevap": "Slop tankı, geminin içindeki kirli su, yağ ve diğer atık sıvıları toplamak için kullanılan tanktır. Bu sıvılar, geminin çevresel standartlara uygun şekilde bertaraf edilmek üzere depolanır."},
    {"soru": "'Yük ve bulaşık tankları'nın planında hangi ölçümler mutlaka belirtilmelidir?", "dogru_cevap": "Her tankın hacmi (m³), derinliği (m), genişliği (m) ve yüksekliği (m) ile birlikte, tankın konumu (geminin hangi bölümünde) ve bağlantı noktaları (boru, valf vb.) belirtilmelidir."},
    {"soru": "MARPOL 73/78 protokolü, gemi atıklarının bertarafı konusunda hangi temel kuralları getirir?", "dogru_cevap": "Atıkların gemide depolanması, taşınması ve bertarafı için sınıflandırma, izleme, kayıt tutma ve raporlama zorunluluğu getirir. Ayrıca, “atık su” ve “atık yağ” gibi belirli atık türlerinin ayrı tanklarda tutulması gerekir."},
    {"soru": "Bir geminin 'slop' tankının derinliği 4 m ise, 200 m³’lük bir hacim elde etmek için genişliği ve yüksekliği ne kadar olmalıdır?", "dogru_cevap": "Hacim = genişlik × yükseklik × derinlik → 200 = G × H × 4 → G × H = 50. Örneğin, genişlik 5 m, yüksekliği 10 m (5×10=50) olabilir."},
    {"soru": "'Yük tankı' ve 'bulaşık tankı' arasındaki temel fark nedir?", "dogru_cevap": "Yük tankı, geminin taşıdığı yükü (örneğin petrol, kimyasal madde) depolamak için kullanılırken, bulaşık tankı geminin içindeki kirli su ve yağ gibi atık sıvıları toplamak için kullanılır."},
    {"soru": "Bir geminin 'slop' tankı 150 m³ kapasiteye sahipse, bu tankın 80% dolu olması durumunda kaç m³ sıvı bulunur?", "dogru_cevap": "150 m³ × 0.80 = 120 m³."},
    {"soru": "'Yük ve bulaşık tankları'nın planında 'No-a' ifadesi ne anlama gelir?", "dogru_cevap": "“No-a” ifadesi, tankın numarasını (No.) ve alanını (a) gösterir. Örneğin, “No-a. 3” tank 3 numaralı tankın alanını ifade eder."},
    {"soru": "MARPOL 73/78 protokolü kapsamında 'atık su' için hangi tank türü kullanılır?", "dogru_cevap": "“Atık su” için “atık su tankı” (waste water tank) kullanılır. Bu tank, geminin içindeki su atıklarını depolamak için ayrılmıştır."},
    {"soru": "Bir geminin 'slop' tankı 250 m³ kapasiteye sahipse, bu tankın 60% dolu olması durumunda kaç m³ sıvı bulunur?", "dogru_cevap": "250 m³ × 0.60 = 150 m³."},
    {"soru": "'Yük tankı'nın planında 'kapasite' ifadesi hangi birimde ölçülür?", "dogru_cevap": "Kapasite metre küp (m³) cinsinden ölçülür."},
    {"soru": "'Bulaşık tankı'nın derinliği 3 m ise, 180 m³’lük bir hacim elde etmek için genişliği ve yüksekliği ne kadar olmalıdır?", "dogru_cevap": "180 = G × H × 3 → G × H = 60. Örneğin, genişlik 6 m, yüksekliği 10 m (6×10=60) olabilir."},
    {"soru": "MARPOL 73/78 protokolü, gemi atıklarının hangi iki ana kategoriye ayrılmasını öngörür?", "dogru_cevap": "“Atık su” (waste water) ve “atık yağ” (waste oil) olmak üzere iki ana kategori."},
    {"soru": "'Yük tankı'nın planında 'derinlik' ölçüsü neden önemlidir?", "dogru_cevap": "Derinlik, tankın iç hacmini belirler ve geminin denge, ballast ve taşıma kapasitesi hesaplamalarında kritik bir parametredir."},
    {"soru": "Bir geminin 'slop' tankı 300 m³ kapasiteye sahipse, bu tankın 90% dolu olması durumunda kaç m³ sıvı bulunur?", "dogru_cevap": "300 m³ × 0.90 = 270 m³."},
    {"soru": "'Yük ve bulaşık tankları'nın planında 'No.ı' ifadesi ne anlama gelir?", "dogru_cevap": "“No.ı” tankın numarasını (No.) gösterir; örneğin “No. 5” beşinci tankı ifade eder."},
    {"soru": "MARPOL 73/78 protokolü, gemi atıklarının hangi iki ana taşıma yöntemini düzenler?", "dogru_cevap": "“Atık su” ve “atık yağ”ın gemi içinde depolanması ve gemiden dışarı atılması (atık su boşaltma, atık yağ boşaltma)."},
    {"soru": "'Bulaşık tankı'nın derinliği 2 m ise, 120 m³’lük bir hacim elde etmek için genişliği ve yüksekliği ne kadar olmalıdır?", "dogru_cevap": "120 = G × H × 2 → G × H = 60. Örneğin, genişlik 4 m, yüksekliği 15 m (4×15=60) olabilir."},
    {"soru": "'Yük tankı'nın planında 'kapasite' ifadesi hangi birimde ölçülür?", "dogru_cevap": "Kapasite metre küp (m³) cinsinden ölçülür."},
    {"soru": "Bir geminin 'slop' tankı 400 m³ kapasiteye sahipse, bu tankın 75% dolu olması durumunda kaç m³ sıvı bulunur?", "dogru_cevap": "400 m³ × 0.75 = 300 m³."},
    {"soru": "'Yük ve bulaşık tankları'nın planında 'derinlik' ölçüsü neden önemlidir?", "dogru_cevap": "Derinlik, tankın iç hacmini belirler ve geminin denge, ballast ve taşıma kapasitesi hesaplamalarında kritik bir parametredir."},
    {"soru": "MARPOL 73/78 protokolü, gemi atıklarının hangi iki ana taşıma yöntemini düzenler?", "dogru_cevap": "“Atık su” ve “atık yağ”ın gemi içinde depolanması ve gemiden dışarı atılması (atık su boşaltma, atık yağ boşaltma)."},
    {"soru": "'Bulaşık tankı'nın derinliği 5 m ise, 250 m³’lük bir hacim elde etmek için genişliği ve yüksekliği ne kadar olmalıdır?", "dogru_cevap": "250 = G × H × 5 → G × H = 50. Örneğin, genişlik 5 m, yüksekliği 10 m (5×10=50) olabilir."},
    {"soru": "'Yük tankı'nın planında 'No-a' ifadesi ne anlama gelir?", "dogru_cevap": "“No-a” ifadesi, tankın numarasını (No.) ve alanını (a) gösterir. Örneğin, “No-a. 3” tank 3 numaralı tankın alanını ifade eder."},
    {"soru": "MARPOL 73/78 protokolü kapsamında 'atık su' için hangi tank türü kullanılır?", "dogru_cevap": "“Atık su” için “atık su tankı” (waste water tank) kullanılır. Bu tank, geminin içindeki su atıklarını depolamak için ayrılmıştır."}
]

# ==========================================
# 2. Model ve Yardımcı Fonksiyonların Tanımı
# ==========================================
hakem_llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.0)

# Önceki adımlarda kurduğumuz hafızadaki retriever ve llm nesnelerini bağlıyoruz
def test_bot(soru):
    docs = retriever.invoke(soru)
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"Bağlam:\n{context}\n\nSoru: {soru}\nCevap:"
    return llm.invoke(prompt).content

print("CANLI TEST BAŞLATILDI. 40 SORU SIRAYLA ANALİZ EDİLİYOR:\n")

# ==========================================
# 3. 🔄 Test Döngüsü
# ==========================================
for idx, senaryo in enumerate(altin_veri_seti):
    soru = senaryo["soru"]
    gercek = senaryo["dogru_cevap"]

    # Hafızadaki hazır yapıdan anlık cevap çekilir
    bot_cevap = test_bot(soru)

    # Hakem İstem Şablonu
    hakem_prompt = f"""Soru: {soru}
Gerçek Doğru Cevap: {gercek}
Öğrenci Cevabı: {bot_cevap}

Yukarıdaki öğrenci cevabına sadece 1 ile 5 arasında bir rakam notu ver ve yanına çok kısa tek cümle gerekçeni yaz.
Örnek format: '4 - Anlamca doğru fakat kelimeler eksik.'"""

    hakem_karari = hakem_llm.invoke(hakem_prompt).content

    # Sonuçları anında ekrana yansıtıyoruz
    print(f"📝 [{idx+1}/40] Soru: {soru}")
    print(f"🤖 Botun Yanıtı: {bot_cevap}")
    print(f"👨‍⚖️ Hakem Kararı: {hakem_karari}")
    print("="*60 + "\n")

    time.sleep(0.5) # Groq rate-limit koruması

print("🏁 40 Soruluk tüm test başarıyla tamamlandı!")


CANLI TEST BAŞLATILDI. 40 SORU SIRAYLA ANALİZ EDİLİYOR:

📝 [1/40] Soru: Metnin başında hangi makamın bildirim yapması gerektiği belirtiliyor?
🤖 Botun Yanıtı: Metnin başında **rapor göndermekle yükümlü kişi** (yani raporu iletmekle sorumlu olan makam) bildirim yapması gerektiği belirtiliyor.
👨‍⚖️ Hakem Kararı: 2 - Kısmen doğru ama tam isim eksik.

📝 [2/40] Soru: Mahkeme kaç üyeden oluşur?
🤖 Botun Yanıtı: 
👨‍⚖️ Hakem Kararı: 1 - Cevap yok, eksik.

📝 [3/40] Soru: Taraflar kaç hakem atar?
🤖 Botun Yanıtı: Taraflar **iki** hakem atar.
👨‍⚖️ Hakem Kararı: 1 - Yanlış, her taraf bir hakem atar.

📝 [4/40] Soru: Üçüncü hakemin atanması kimlerin mutabakatıyla gerçekleşir?
🤖 Botun Yanıtı: 
👨‍⚖️ Hakem Kararı: 1 - Cevap yok, eksik.

📝 [5/40] Soru: İkinci hakemin tayininden sonra kaç gün içinde başkan tayin edilemezse ne olur?
🤖 Botun Yanıtı: 
👨‍⚖️ Hakem Kararı: 1 - Cevap yok, eksik.

📝 [6/40] Soru: Tamamlayıcı raporun gönderilmesi hangi şartlara bağlıdır?
🤖 Botun Yanıtı: **Tamamlayıcı raporun gönderil

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-20b` in organization `org_01m4nzqfgbed79qcpzj8ksat6x` service tier `on_demand` on tokens per day (TPD): Limit 200000, Used 197357, Requested 2786. Please try again in 1m1.776s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

***TEST SONUCU:***

*Toplam Soru: 40 adet*

* Doğru: 18 adet
* Yanlış: 8 adet
* Boş: 4 adet
* Esik/Kısmen doğru: 10 adet
----------------------------
* 5 puan: 18 adet
* 4 puan: 2 adet
* 3 puan: 6 adet
* 2 puan: 3 adet
* 1 puan: 11 adet

**!!! Test sonucuna göre , prompt güncelleme ve chunk değer değişiklikleri ? yapalım !!!**

10.Hücre - STREAMLIT arayüz hazırlık ve Tünelleme

In [ ]:
%%writefile app.py
import os, streamlit as st
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

st.set_page_config(layout="wide")

# 🎨 Deniz Temizliği ve Denizcilik Teması Renk Ayarları (CSS)
st.markdown("""
    <style>
    .stApp { background-color: #f0f5f9; }
    h1 { color: #0a4b7c !important; }
    .stChatMessage { border-radius: 12px; padding: 10px; margin-bottom: 8px; }
    div[data-testid="stChatMessageUser"] { background-color: #e0f2fe !important; border-left: 5px solid #0284c7; }
    div[data-testid="stChatMessageAssistant"] { background-color: #ffffff !important; border-left: 5px solid #0f172a; border: 1px solid #e2e8f0; }
    </style>
""", unsafe_allow_html=True)

st.title("🌊 MARPOL RAG Chatbot")

@st.cache_resource
def init_fast_rag(uploaded_file):
    temp_path = "temp.pdf"
    with open(temp_path, "wb") as f:
        f.write(uploaded_file.getbuffer())

    loader = PyPDFLoader(temp_path)
    splits = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=200).split_documents(loader.load())

    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vectorstore = InMemoryVectorStore.from_documents(splits, embeddings)
    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

    # LLM MODEL: Saniyede 900+ kelime üreten kararlı yapı
    llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0.1)
    prompt = ChatPromptTemplate.from_template("Bağlam:\n{context}\n\nSoru: {input}\nCevap:")

    chain = (
        {"context": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)), "input": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain, len(splits)

uploaded_file = st.file_uploader("⚓ Analiz edilecek denizcilik belgesini buraya bırakın", type=["pdf"])

if uploaded_file:
    rag_chain, split_count = init_fast_rag(uploaded_file)
    st.success(f"💧 Belge saniyeler içinde hafızaya alındı ({split_count} parça)! Yapay zeka hazır.")

    if "messages" not in st.session_state: st.session_state.messages = []
    for msg in st.session_state.messages: st.chat_message(msg["role"]).write(msg["content"])

    if user_query := st.chat_input("Sorunuzu buraya yazın..."):
        st.chat_message("user").write(user_query)
        st.session_state.messages.append({"role": "user", "content": user_query})

        with st.chat_message("assistant"):
            with st.spinner("Metin derinliklerinde aranıyor..."):
                response = rag_chain.invoke(user_query)
                st.write(response)
        st.session_state.messages.append({"role": "assistant", "content": response})


Overwriting app.py


11.Hücre - Sunucu çalıştır

In [ ]:
import threading
from google.colab import output

# 1. Port çakışmalarını tamamen temizlemek için uygulamayı 8540 portunda başlat
def run_streamlit():
    !streamlit run app.py \
        --server.port 8540 \
        --server.enableCORS=false \
        --server.enableXsrfProtection=false \
        --server.maxUploadSize=500 \
        --server.fileWatcherType=none

threading.Thread(target=run_streamlit).start()
print("⚡ Akış kanalı optimize edildi. Uygulama hücre altına yükleniyor...")

# 2. Google'ın Axios ağ hatalarını engelleyen resmi iç köprüsünü çalıştırıyoruz
# height=800 parametresi dosyaları rahat yüklemeniz için ekranı daha da büyütür.
output.serve_kernel_port_as_iframe(8540, height=800)


⚡ Akış kanalı optimize edildi. Uygulama hücre altına yükleniyor...


<IPython.core.display.Javascript object>

14.Hücre - COLAB ile arayüz oluşturma (streamlit alternatif )

In [ ]:
#!pip install -q gradio
#!pip install -qU sentence-transformers



15.Hücre FAISS yükleme

16.Hücre - COLAB gradio yükleme (arayüz oluşturma)